# Multimodal Late Fusion & Adaptive Recommendation Engine
### *Fusion Layer · Cognitive Load × Student Engagement*

---

| Field | Details |
|-------|---------|
| **Project** | Non-Invasive Cognitive Load and Student Engagement Detection |
| **Author** | Yasini Mandara Karunanayake |
| **RGU ID** | 2313473 **IIT ID** 20221151 |


---

## Notebook Overview

This notebook implements the **late fusion layer** and **rule-based adaptive recommendation engine** described in the thesis.
It combines the class probability outputs of the two independently trained branch models into a nine-state combined representation and maps each state to a specific pedagogical action.

| Step | Description |
|------|-------------|
| 1 | Setup, imports & shared configuration |
| 2 | Load pre-trained models & test data |
| 3 | Base model predictions (CL + SE branches) |
| 4 | Late fusion layer (confidence-weighted probability averaging) |
| 5 | Fusion evaluation metrics (Accuracy, Macro F1, Cohen's Kappa) |
| 6 | Ablation study (CL-only vs SE-only vs fusion) |
| 7 | Multimodal fusion quality evaluation |
| 8 | Probability calibration analysis |
| 9 | Inference latency benchmark |
| 10 | SHAP interpretability (eye-metric model) |
| 11 | Rule-based adaptive action engine |
| 12 | End-to-end inference demo |
| 13 | Pipeline summary & export |


In [ ]:
!pip install -q scikit-image joblib
print('Dependencies ready')

Dependencies ready


In [ ]:
import os

BASE_DIR = '/content'

# List all files this notebook depends on
required_files = [
    'best_model.pkl',
    'best_model_scaler.pkl',
    'best_engagement_model.pkl',
    'X_test.csv',
    'X_test_hog.npy',
    'y_test.csv',
    'y_test.npy',
]

# Check each required file exists and print its size
print('Checking required files:')
all_found = True
for fname in required_files:
    fpath = os.path.join(BASE_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f'  {fname:<35s}  {size_kb:>8.1f} KB')
    else:
        print(f'  {fname:<35s}  NOT FOUND')
        all_found = False

print()
# Report overall status after scanning all files
if all_found:
    print('All files found')
else:
    print('Some files are missing')


Checking required files:
  best_model.pkl                            1.3 KB
  best_model_scaler.pkl                     1.0 KB
  best_engagement_model.pkl              7075.7 KB
  X_test.csv                               16.1 KB
  X_test_hog.npy                         2873.5 KB
  y_test.csv                                1.8 KB
  y_test.npy                                1.8 KB

All files found


## 1. Setup and Imports

Import all required libraries and define the shared configuration used throughout this notebook:
label encodings, the nine combined cognitive-engagement states, and the colour palette for plots.

In [ ]:
# Standard library and third-party imports
import warnings, joblib, json, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Scikit-learn evaluation metrics
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, confusion_matrix,
    accuracy_score, f1_score, cohen_kappa_score
)

# Suppress warnings and fix random seed for reproducibility
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)
RANDOM_STATE = 42

# Directory where all output files will be saved
OUTPUT_DIR = '/content'

# Shared label space used by both the CL and SE models
LEVELS      = ['low', 'medium', 'high']
LEVEL_ENC   = {l: i for i, l in enumerate(LEVELS)}
LEVEL_DEC   = {i: l for l, i in LEVEL_ENC.items()}
CLASS_NAMES = ['Low', 'Medium', 'High']
PALETTE     = {'low': '#e74c3c', 'medium': '#f39c12', 'high': '#2ecc71'}

# Nine combined states formed by crossing CL level (rows) with SE level (columns)
# Each combination maps to a named pedagogical state
COMBINED_STATES = {
    ('high',   'low'):    'Overloaded',
    ('high',   'medium'): 'Struggling',
    ('high',   'high'):   'Trying Hard',
    ('medium', 'low'):    'Distracted',
    ('medium', 'medium'): 'Focused',
    ('medium', 'high'):   'Engaged',
    ('low',    'low'):    'Bored',
    ('low',    'medium'): 'Relaxed',
    ('low',    'high'):   'Unchallenged',
}
# Sort state names alphabetically for consistent indexing throughout the notebook
STATE_NAMES = sorted(set(COMBINED_STATES.values()))
STATE_ENC   = {s: i for i, s in enumerate(STATE_NAMES)}
STATE_DEC   = {i: s for s, i in STATE_ENC.items()}


print(f'Classes         : {CLASS_NAMES}')
print(f'Combined states : {STATE_NAMES}')


Classes         : ['Low', 'Medium', 'High']
Combined states : ['Bored', 'Distracted', 'Engaged', 'Focused', 'Overloaded', 'Relaxed', 'Struggling', 'Trying Hard', 'Unchallenged']


## 2.Load Pre-Trained Models and Test Data

Load the four pre-trained artefacts exported by the CL and SE training notebooks, then load the real held-out test data for both branches.

### 2.1 Load Pre-Trained Models

In [ ]:
PICKLE_DIR = Path('/content')

# CL branch: Logistic Regression classifier and its fitted StandardScaler
lr_cl      = joblib.load(PICKLE_DIR / 'best_model.pkl')
cog_scaler = joblib.load(PICKLE_DIR / 'best_model_scaler.pkl')

# SE branch: student engagement classifier
xgb_se     = joblib.load(PICKLE_DIR / 'best_engagement_model.pkl')

print("All models loaded successfully")


All models loaded successfully


### 2.2 Load Real Test Data

Load the held-out test data for both branches:
- **CL branch** — `X_test.csv` contains 4 eye-metric features; scaled by `cognitive_scaler`
- **SE branch** — `X_test_hog.npy` contains 1,764 HOG features

The two datasets have different sizes (SE: 417, CL: 600). Both are truncated to the smaller size so all downstream fusion operations work on aligned arrays.

In [ ]:
# SE real data + ground truth
X_se_raw    = np.load('X_test_hog.npy')
X_se_scaled = hog_scaler.transform(X_se_raw)
y_true_se   = np.load('y_test.npy')          # ground-truth SE labels (417,)
print(f'SE data  : {X_se_raw.shape}   labels: {y_true_se.shape}  classes: {np.unique(y_true_se)}')

# CL real data + ground truth
X_cl_df     = pd.read_csv('X_test.csv')
# Convert the CSV dataframe to a NumPy array for model input
X_cl_raw    = X_cl_df.values
# Scale eye-metric features using the scaler fitted during CL model training
X_cl_scaled = cog_scaler.transform(X_cl_raw)
y_true_cl   = pd.read_csv('y_test.csv').iloc[:, 0].values  # ground-truth CL labels (600,)
print(f'CL data  : {X_cl_raw.shape}   labels: {y_true_cl.shape}  classes: {np.unique(y_true_cl)}')

# Align everything to the smaller dataset (SE=417, CL=600)
N_SAMPLES   = min(len(X_cl_scaled), len(X_se_scaled))
X_cl_scaled = X_cl_scaled[:N_SAMPLES]
X_se_scaled = X_se_scaled[:N_SAMPLES]
y_true_cl   = y_true_cl[:N_SAMPLES]
y_true_se   = y_true_se[:N_SAMPLES]
print(f'\nAligned N_SAMPLES = {N_SAMPLES}')
print(f'CL: X={X_cl_scaled.shape}  y={y_true_cl.shape}')
print(f'SE: X={X_se_scaled.shape}  y={y_true_se.shape}')


SE data  : (417, 1764)   labels: (417,)  classes: [0 1 2]
CL data  : (600, 4)   labels: (600,)  classes: [0 1 2]

Aligned N_SAMPLES = 417
CL: X=(417, 4)  y=(417,)
SE: X=(417, 1764)  y=(417,)
